In [5]:
import os
import json
import pandas as pd
import requests
import ast
import re
import time
from tqdm import tqdm
from openai import OpenAI
from pypfopt import expected_returns, risk_models, EfficientFrontier
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

# 配置 API Key 和 URL
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY", "")
BASE_URL = (
    "https://api.polygon.io/v2/aggs/ticker/{ticker}/range/"
    "{multiplier}/{timespan}/{from_}/{to}"
)

In [18]:
def generate_portfolios():
    client = OpenAI(api_key=OPENAI_API_KEY)
    prompt = (
        "You are an expert portfolio construction advisor. "
        "Based on the past year of US stock performance, generate a portfolio suggestions "
        "that includes 15-30 TICKERS ONLY based on current market performance"
        "Output must be a pure JSON list, each element containing 'name'."
        "DO NOT INCLUDE ANY EXPLANATIONS, JSON ONLY"
    )
    resp = client.responses.create(
        model="gpt-5", 
        tools=[{"type": "web_search_preview"}], 
        input=prompt
    )
    raw = resp.output_text
    txt_path = os.path.join("D:/my-fin-project/txt_save", "portfolios.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(raw)
    print("原始组合已保存到 txt_save/portfolios.txt")
    return convert_portfolios_txt_to_json(txt_path, "D:\my-fin-project\json_save\portfolios.json")

# ---------------------------------------------------------------------------------------------------------------------

def convert_portfolios_txt_to_json(input_path: str, output_path: str):
    def _strip_trailing_commas(s: str) -> str:
        s = s.strip()
        s = re.sub(r',(\s*[}\]])', r'\1', s)
        s = s.rstrip(", \t\r\n")
        return s

    with open(input_path, "r", encoding="utf-8-sig") as f:
        raw = f.read().strip()
    if not raw:
        data = []
    else:
        try:
            obj = json.loads(_strip_trailing_commas(raw))
            data = obj if isinstance(obj, list) else [obj]
        except json.JSONDecodeError:
            items = []
            for line in raw.splitlines():
                s = line.strip()
                if not s or s.startswith("//") or s.startswith("#"):
                    continue
                if s.endswith(","):
                    s = s[:-1].rstrip()
                try:
                    items.append(json.loads(_strip_trailing_commas(s)))
                except json.JSONDecodeError:
                    pass
            if items:
                data = items
            else:
                objs = re.findall(r'\{(?:[^{}]|(?R))*\}', raw, flags=re.DOTALL)
                if objs:
                    joined = "[" + ",".join(o.rstrip(", \t\r\n") for o in objs) + "]"
                    try:
                        data = json.loads(_strip_trailing_commas(joined))
                    except json.JSONDecodeError:
                        raise ValueError("无法识别文件格式")
                else:
                    raise ValueError("无法识别文件格式")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"解析完成，已保存到 {output_path}")
    return data

In [19]:
generate_portfolios()

原始组合已保存到 txt_save/portfolios.txt
解析完成，已保存到 D:\my-fin-project\json_save\portfolios.json


[{'name': 'PLTR'},
 {'name': 'GEV'},
 {'name': 'VST'},
 {'name': 'CEG'},
 {'name': 'AVGO'},
 {'name': 'NVDA'},
 {'name': 'SMCI'},
 {'name': 'AMD'},
 {'name': 'MU'},
 {'name': 'STX'},
 {'name': 'WDC'},
 {'name': 'HWM'},
 {'name': 'JBL'},
 {'name': 'AXON'},
 {'name': 'ORCL'},
 {'name': 'CRWD'},
 {'name': 'MSFT'},
 {'name': 'GOOGL'},
 {'name': 'META'},
 {'name': 'AMZN'},
 {'name': 'LLY'},
 {'name': 'GE'},
 {'name': 'NRG'},
 {'name': 'RCL'}]

In [15]:
def load_portfolios(json_file=None):
    if not json_file:
        json_file = os.path.join("D:/my-fin-project/json_save", "portfolios.json")
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    tickers = [item["name"] for item in data]
    return tickers

In [16]:
def fetch_and_save(tickers, start, end, filename, multiplier=1, timespan="day", sleep_time=1):  
    records = []
    for t in tqdm(tickers, desc="Fetching data"):
        url = BASE_URL.format(ticker=t, multiplier=multiplier, timespan=timespan, from_=start, to=end)
        params = {
            "adjusted": True,
            "sort": "asc",
            "limit": 50000,
            "apiKey": POLYGON_API_KEY
        }
        try:
            resp = requests.get(url, params=params)
            resp.raise_for_status()
            results = resp.json().get("results", [])
            if not results:
                print(f"⚠️ {t} 无历史数据，跳过。")
                continue
            for item in results:
                records.append({
                    "date": pd.to_datetime(item["t"], unit="ms"),
                    "ticker": t,
                    "close": item["c"]
                })
        except Exception as e:
            print(f"❌ 获取 {t} 数据失败：{e}")
            continue
        time.sleep(sleep_time)  # 限速保护，避免触发 API 限额

    if not records:
        print("⚠️ 所有 ticker 数据为空，未生成 CSV 文件。")
        return None

    # 构建宽表格式（日期为索引，ticker 为列）
    df = pd.DataFrame(records).pivot(index="date", columns="ticker", values="close")
    df.sort_index(inplace=True)
    path = os.path.join("D:/my-fin-project/csv_save", filename)
    df.to_csv(path)
    print(f"✅ 行情数据已保存到 csv_save/{filename}，共 {df.shape[0]} 行 × {df.shape[1]} 列")
    return path

In [17]:
tickers = load_portfolios()
fetch_and_save(tickers=tickers, start="2024-07-25", end="2025-07-25", filename='test_portfolio.csv')

Fetching data: 100%|██████████| 25/25 [00:47<00:00,  1.91s/it]

✅ 行情数据已保存到 csv_save/test_portfolio.csv，共 251 行 × 25 列


'D:/my-fin-project/csv_save\\test_portfolio.csv'

In [8]:
def analyze_performance(csv_file):
    # Read in price data
    df = pd.read_csv(csv_file, parse_dates=True, index_col="date")
    # Calculate expected returns and sample covariance
    mu = expected_returns.mean_historical_return(df)
    S = risk_models.sample_cov(df)

    # Optimize for maximal Sharpe ratio
    ef = EfficientFrontier(mu, S)
    raw_weights = ef.max_sharpe()
    cleaned_weights = ef.clean_weights()
    ef.save_weights_to_file("weights.csv")  # saves to file
    print(cleaned_weights)
    ef.portfolio_performance(verbose=True)
    # Calculate expected returns and sample covariance
    latest_prices = get_latest_prices(df)
    da = DiscreteAllocation(cleaned_weights, latest_prices, total_portfolio_value=25741.99)
    allocation, leftover = da.greedy_portfolio()
    print("Discrete allocation:", allocation)
    print("Funds remaining: ${:.2f}".format(leftover))


In [16]:
analyze_performance(r'D:\my-fin-project\csv_save\prices_20250808-192013.csv')

OrderedDict([('AAPL', 0.0), ('AMZN', 0.0), ('BAC', 0.0), ('CSCO', 0.15094), ('CVX', 0.0), ('DIS', 0.0), ('GOOGL', 0.0), ('HD', 0.0), ('INTC', 0.0), ('JNJ', 0.0), ('JPM', 0.0), ('KO', 0.0), ('META', 0.08796), ('MSFT', 0.0), ('NFLX', 0.28824), ('NVDA', 0.0), ('PEP', 0.0), ('PG', 0.0), ('T', 0.40003), ('TSLA', 0.0), ('UNH', 0.0), ('V', 0.0), ('VZ', 0.0), ('WMT', 0.07283), ('XOM', 0.0)])
Expected annual return: 57.9%
Annual volatility: 17.9%
Sharpe Ratio: 3.24
Discrete allocation: {'T': 366, 'NFLX': 6, 'CSCO': 55, 'META': 3, 'WMT': 18}
Funds remaining: $403.66


In [22]:
import pandas as pd
import numpy as np
def risk_analysis(csv_file):
    # === 参数设定 ===
    confidence_level = 0.95
    portfolio_value = 1_000_000  # 每个股票假设管理100万元

    # === Step 1: 读取数据 ===
    df = pd.read_csv(csv_file, parse_dates=["date"])
    df = df.set_index("date").sort_index()

    # === Step 2: 计算所有ticker的收益率 ===
    returns = df.pct_change().dropna()

    # === Step 3: 批量计算 VaR 和 CVaR ===
    result = []

    for ticker in returns.columns:
        r = returns[ticker].dropna()
        var = np.percentile(r, (1 - confidence_level) * 100)
        cvar = r[r <= var].mean()
        result.append({
            "Ticker": ticker,
            f"VaR_{int(confidence_level*100)}": var,
            f"CVaR_{int(confidence_level*100)}": cvar,
            "VaR_amount": -var * portfolio_value,
            "CVaR_amount": -cvar * portfolio_value
        })

    # === Step 4: 输出结果表格 ===
    result_df = pd.DataFrame(result)
    result_df[f"VaR_{int(confidence_level*100)}"] = result_df[f"VaR_{int(confidence_level*100)}"].map(lambda x: f"{x:.2%}")
    result_df[f"CVaR_{int(confidence_level*100)}"] = result_df[f"CVaR_{int(confidence_level*100)}"].map(lambda x: f"{x:.2%}")
    print(result_df)


In [23]:
risk_analysis(r'D:\my-fin-project\csv_save\test_portfolio.csv')

   Ticker   VaR_95  CVaR_95     VaR_amount    CVaR_amount
0    AMAT   -5.02%   -7.39%   50179.918217   73942.296318
1    AMZN   -3.19%   -4.80%   31855.336776   47966.551326
2     ARM   -6.44%   -8.59%   64427.474215   85875.021802
3    AVGO   -5.01%   -7.84%   50075.697659   78447.750833
4    COST   -1.84%   -3.19%   18429.311438   31867.335942
5    CRWD   -4.16%   -6.20%   41640.495731   61973.155899
6      GE   -2.87%   -5.16%   28678.569389   51583.877564
7   GOOGL   -3.50%   -4.64%   35044.069706   46381.436135
8    KLAC   -4.24%   -7.26%   42434.319938   72632.060379
9     LLY   -3.61%   -5.46%   36088.037634   54604.973662
10   LRCX   -4.59%   -7.51%   45898.851633   75101.434716
11   META   -3.29%   -4.69%   32891.345715   46868.871298
12   MSFT   -2.19%   -3.45%   21918.294128   34545.383707
13     MU   -4.84%   -9.37%   48369.718821   93722.900307
14   NFLX   -2.73%   -4.25%   27345.823722   42499.360128
15    NOW   -3.07%   -5.34%   30661.035804   53408.532157
16   NVDA   -5

In [1]:
import os
import requests
from openai import OpenAI

def check_openai():
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        return "❌ OpenAI API key not found in environment variables"
    try:
        client = OpenAI(api_key=key)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "Test"}],
            max_tokens=5
        )
        return f"✅ OpenAI API is working (model: gpt-4o-mini, output: {resp.choices[0].message.content})"
    except Exception as e:
        return f"❌ OpenAI API error: {e}"

def check_polygon():
    key = os.getenv("POLYGON_API_KEY")
    if not key:
        return "❌ Polygon API key not found in environment variables"
    try:
        url = f"https://api.polygon.io/v1/marketstatus/now?apiKey={key}"
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            return f"✅ Polygon API is working (status: {r.json().get('market')})"
        else:
            return f"❌ Polygon API HTTP error: {r.status_code} - {r.text}"
    except Exception as e:
        return f"❌ Polygon API error: {e}"

def check_deepseek():
    key = os.getenv("DEEPSEEK_API_KEY")
    if not key:
        return "❌ DeepSeek API key not found in environment variables"
    try:
        client = OpenAI(
            api_key=key,
            base_url="https://api.deepseek.com"  # DeepSeek API 入口
        )
        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": "Test"}],
            max_tokens=5
        )
        return f"✅ DeepSeek API is working (model: deepseek-chat, output: {resp.choices[0].message.content})"
    except Exception as e:
        return f"❌ DeepSeek API error: {e}"

if __name__ == "__main__":
    print(check_openai())
    print(check_polygon())
    print(check_deepseek())


✅ OpenAI API is working (model: gpt-4o-mini, output: Test received! How can)
✅ Polygon API is working (status: extended-hours)
✅ DeepSeek API is working (model: deepseek-chat, output: Hello! It looks like)


In [2]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

models = client.models.list()
for m in models.data:
    print(m.id)


deepseek-chat
deepseek-reasoner
